# 時系列予測

特徴量ベースと系列モデルの使い分け、時系列の検証とリーク回避を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/deep-learning/08-time-series`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# 複数系列（store_A / store_B）の日次売上サンプルデータを生成します
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
dates = pd.date_range("2025-01-01", periods=14, freq="D")
df = pd.DataFrame({
    "id": np.repeat(["store_A", "store_B"], len(dates)),
    "date": np.tile(dates, 2),
    "y": (100 + np.tile(np.arange(len(dates)) * 2.0, 2) + rng.normal(0, 4.0, size=len(dates) * 2)).round(2),
})

## 2 つのアプローチ

時系列予測には大きく 2 系統があります。

- **特徴量ベース**：ラグ・移動平均・季節性などを特徴量にして 勾配ブースティング（GBDT） で予測。
  実務・コンペで非常に強く、まず試す価値がある
- **系列モデル**：RNN/LSTM や Transformer 系で系列をそのまま学習。長期依存や複雑なパターン向け

多くのテーブル系時系列コンペでは、**特徴量ベース + GBDT** が依然として強力です。

## 時系列特徴量

In [ ]:
# ラグ特徴と移動統計（グループごと・時間順を保つ）
df = df.sort_values(["id", "date"])
df["lag_1"] = df.groupby("id")["y"].shift(1)
df["roll_mean_7"] = df.groupby("id")["y"].transform(lambda s: s.shift(1).rolling(7).mean())

In [ ]:
df[["id", "date", "y", "lag_1", "roll_mean_7"]].dropna().head(8)

`shift(1)` を挟むことで、**予測時点で未来の値を使わない**ようにします。

## 検証はリークとの戦い

> **Warning:** 時系列で通常の K-Fold（ランダム分割）を使うと、**未来で過去を予測**してしまいデータリークに なります。必ず時系列分割（過去で学習し未来で検証）を使い、特徴量の集計も 「その時点までの情報だけ」で作ります。

- 移動統計・ラグは必ず**過去方向**のみ（未来を覗かない）
- 検証期間は本番の予測期間と長さ・位置をそろえる
- リークの典型：未来の値を含む集計、`shift` 忘れ、全期間での標準化

> **Tip:** まず「特徴量ベース + GBDT + 時系列分割」で堅いベースラインを作り、必要なら系列モデルを 足してアンサンブルする、という順序が安全です。

## 次のステップ

最後に、Kaggle 環境での GPU/TPU 活用・混合精度・メモリ最適化を学びます。